Project Title:  **E-Commerce Sales Analysis Using PySpark**

Objective:

*   Tools Used: Pyspark
*   Dataset source: Kaggle + slightly modified for data quality demo

**Environment setup**

In [ ]:
#pyspark installation
!pip install pyspark

In [ ]:
#creating Spark Session
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Task1 Big data analysis').getOrCreate()
spark

**Data loading**

In [ ]:
#loading csv file and creating the dataframe
df = spark.read.csv('/content/OnlineRetail_Analysis.csv', header=True, inferSchema=True)
#header=True considers first row as headers
#inferSchema=True helps identify datatypes of column based on the data present in it.

In [ ]:
#display rows of the table
df.show()

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|   17850.0|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|   17850.0|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
|   536365|    22752|SET 7 BABUSHKA NE...|       2|2010-12-01 08:26:00|     7.65|   17850.0|United Kingdom|
|   536365|    21730|GLASS S

The dataset contains transactional e-commerce data including invoice details, product information, customer id, quantity, unit price, country, and order date.

**Data Understanding & Profiling**

In [ ]:
#schema of the table
df.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)



In [ ]:
#since inferSchema considered CustomerID as double, explicilty changing it to string
from pyspark.sql.functions import col
df = df.withColumn('CustomerID', col('CustomerID').cast('string'))

# Display the updated schema to confirm the change
df.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Country: string (nullable = true)



In [ ]:
#datatypes of each column
df.dtypes

[('InvoiceNo', 'string'),
 ('StockCode', 'string'),
 ('Description', 'string'),
 ('Quantity', 'int'),
 ('InvoiceDate', 'timestamp'),
 ('UnitPrice', 'double'),
 ('CustomerID', 'string'),
 ('Country', 'string')]

In [ ]:
#total no of rows
total_count = df.count()
total_count

2329164

In [ ]:
#distinct count
distinct_count = df.distinct().count()
distinct_count

2242062

*   Distinct count is not same as total count, this indicates duplicates.



In [ ]:
#No of duplicates
dup_count = total_count - distinct_count
dup_count

88246

In [ ]:
#show duplicates
df.groupBy(df.columns).count().filter("count > 1").show(10)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+-----+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|count|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+-----+
|   536637|    22867|HAND WARMER BIRD ...|       1|2010-12-02 11:35:00|      2.1|   18041.0|United Kingdom|    2|
|   541266|    22567|20 DOLLY PEGS RET...|       5|2011-01-16 16:25:00|     1.25|   15673.0|United Kingdom|    2|
|   536592|    22582|PACK OF 6 SWEETIE...|       2|2010-12-01 17:06:00|     5.06|      NULL|United Kingdom|    2|
|   536592|   90124B|BLUE MURANO TWIST...|       1|2010-12-01 17:06:00|    10.17|      NULL|United Kingdom|    2|
|   536864|   72349b|SET/6 PURPLE BUTT...|       1|2010-12-03 11:27:00|     4.21|      NULL|United Kingdom|    2|
|   536876|    21381|MINI WOODEN HAPPY...|       2|2010-12-03 11:36:00|     3.36|      N

In [ ]:
#numerical summary
df.describe().show()

+-------+------------------+------------------+--------------------+------------------+-----------------+------------------+-----------+
|summary|         InvoiceNo|         StockCode|         Description|          Quantity|        UnitPrice|        CustomerID|    Country|
+-------+------------------+------------------+--------------------+------------------+-----------------+------------------+-----------+
|  count|           2441135|           2441135|             2434480|           2169817|          2169817|           1628945|    2441135|
|   mean| 559948.4881821201|27667.354933365164|             20713.0|6.5664486912951645|2.276260397081338|15289.401322328255|       NULL|
| stddev|13425.117501712477|16873.897301754474|                NULL|240.66272467699437|90.62351936320817|1713.8220260335822|       NULL|
|    min|            536365|             10002|"ASSORTED FLOWER ...|            -80995|        -11062.06|           12346.0|  Australia|
|    max|           C581569|             

*   Negative values can be observed in UnitPrice and Quantity.
*   Null values can be observed in Quantity and CustomerID
*   Unspecified country seen





In [ ]:
#checking if InvoiceDate has any null values since its summary is not present in describe()
df.filter(df.InvoiceDate.isNull()).count()

0

In [ ]:
#count of negative and zero UnitPrice
df.filter(df.UnitPrice <= 0).count()

550417

In [ ]:
#displaying rows with negative and zero UnitPrice
df.filter(df.UnitPrice <= 0).show(10)

+---------+---------+-----------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+-----------+--------+-------------------+---------+----------+--------------+
|   536414|    22139|       NULL|      56|2010-12-01 11:52:00|      0.0|      NULL|United Kingdom|
|   536545|    21134|       NULL|       1|2010-12-01 14:32:00|      0.0|      NULL|United Kingdom|
|   536546|    22145|       NULL|       1|2010-12-01 14:33:00|      0.0|      NULL|United Kingdom|
|   536547|    37509|       NULL|       1|2010-12-01 14:33:00|      0.0|      NULL|United Kingdom|
|   536549|   85226A|       NULL|       1|2010-12-01 14:34:00|      0.0|      NULL|United Kingdom|
|   536550|    85044|       NULL|       1|2010-12-01 14:34:00|      0.0|      NULL|United Kingdom|
|   536552|    20950|       NULL|       1|2010-12-01 14:34:00|      0.0|      NULL|United Kingdom|
|   536553

In [ ]:
#count of negative and zero quantities
df.filter(df.Quantity <= 0).count()

574700

In [ ]:
#displaying rows with negative and zero quantities
df.filter(df.Quantity <= 0).show(10)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|  C536379|        D|            Discount|      -1|2010-12-01 09:41:00|     27.5|   14527.0|United Kingdom|
|  C536383|   35004C|SET OF 3 COLOURED...|      -1|2010-12-01 09:49:00|     4.65|   15311.0|United Kingdom|
|  C536391|    22556|PLASTERS IN TIN C...|     -12|2010-12-01 10:24:00|     1.65|   17548.0|United Kingdom|
|  C536391|    21984|PACK OF 12 PINK P...|     -24|2010-12-01 10:24:00|     0.29|   17548.0|United Kingdom|
|  C536391|    21983|PACK OF 12 BLUE P...|     -24|2010-12-01 10:24:00|     0.29|   17548.0|United Kingdom|
|  C536391|    21980|PACK OF 12 RED RE...|     -24|2010-12-01 10:24:00|     0.29|   17548.0|United Kingdom|
|  C536391|    21484|CHICK G

In [ ]:
#count of null values in Quantity
df.filter(df.Quantity.isNull()).count()

271318

In [ ]:
#displaying rows with null quantities
df.filter(df.Quantity.isNull()).show(10)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536365|   84029G|KNITTED UNION FLA...|    NULL|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
|   536366|    22633|HAND WARMER UNION...|    NULL|2010-12-01 08:28:00|     1.85|   17850.0|United Kingdom|
|   536367|    22745|POPPY'S PLAYHOUSE...|    NULL|2010-12-01 08:34:00|      2.1|   13047.0|United Kingdom|
|   536367|    22749|FELTCRAFT PRINCES...|    NULL|2010-12-01 08:34:00|     3.75|   13047.0|United Kingdom|
|   536367|    22622|BOX OF VINTAGE AL...|    NULL|2010-12-01 08:34:00|     9.95|   13047.0|United Kingdom|
|   536367|    21755|LOVE BUILDING BLO...|    NULL|2010-12-01 08:34:00|     5.95|   13047.0|United Kingdom|
|   536367|    48187| DOORMA

In [ ]:
#count of null values in CustomerID
df.filter(df.CustomerID.isNull()).count()

589332

In [ ]:
#displaying rows with blank CustomerID's
df.filter(df.CustomerID.isNull()).show(10)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536414|    22139|                NULL|      56|2010-12-01 11:52:00|      0.0|      NULL|United Kingdom|
|   536544|    21773|DECORATIVE ROSE B...|       1|2010-12-01 14:32:00|     2.51|      NULL|United Kingdom|
|   536544|    21774|DECORATIVE CATS B...|       2|2010-12-01 14:32:00|     2.51|      NULL|United Kingdom|
|   536544|    21786|   POLKADOT RAIN HAT|       4|2010-12-01 14:32:00|     0.85|      NULL|United Kingdom|
|   536544|    21787|RAIN PONCHO RETRO...|       2|2010-12-01 14:32:00|     1.66|      NULL|United Kingdom|
|   536544|    21790|  VINTAGE SNAP CARDS|       9|2010-12-01 14:32:00|     1.66|      NULL|United Kingdom|
|   536544|    21791|VINTAGE

In [ ]:
#No of rows with blank country
df.filter(df.Country.isNull()).count()

0

In [ ]:
#distinct country
df.select('Country').distinct().count()

38

In [ ]:
#show distinct country
df.select('Country').distinct().show(38)

+--------------------+
|             Country|
+--------------------+
|              Sweden|
|           Singapore|
|             Germany|
|                 RSA|
|              France|
|              Greece|
|  European Community|
|             Belgium|
|             Finland|
|               Malta|
|         Unspecified|
|               Italy|
|                EIRE|
|           Lithuania|
|              Norway|
|               Spain|
|             Denmark|
|           Hong Kong|
|             Iceland|
|              Israel|
|     Channel Islands|
|                 USA|
|              Cyprus|
|        Saudi Arabia|
|         Switzerland|
|United Arab Emirates|
|              Canada|
|      Czech Republic|
|              Brazil|
|             Lebanon|
|               Japan|
|              Poland|
|            Portugal|
|           Australia|
|             Austria|
|             Bahrain|
|      United Kingdom|
|         Netherlands|
+--------------------+



In [ ]:
#Unspecified country
df.filter(df.Country == 'Unspecified').count()

2049

In [ ]:
#displaying rows with Unspecified country
df.filter(df.Country == 'Unspecified').show(10)

+---------+---------+--------------------+--------+-------------------+---------+----------+-----------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|    Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+-----------+
|   549687|    20685|DOORMAT RED RETRO...|       2|2011-04-11 13:29:00|     7.95|   12363.0|Unspecified|
|   549687|    22691|DOORMAT WELCOME S...|       2|2011-04-11 13:29:00|     7.95|   12363.0|Unspecified|
|   549687|    48116|DOORMAT MULTICOLO...|       2|2011-04-11 13:29:00|     7.95|   12363.0|Unspecified|
|   549687|    21213|PACK OF 72 SKULL ...|      24|2011-04-11 13:29:00|     0.55|   12363.0|Unspecified|
|   549687|    21977|PACK OF 60 PINK P...|      24|2011-04-11 13:29:00|     0.55|   12363.0|Unspecified|
|   549687|    21976|PACK OF 60 MUSHRO...|      24|2011-04-11 13:29:00|     0.55|   12363.0|Unspecified|
|   549687|    21212|PACK OF 72 RETROS...|      24|2011

In [ ]:
#The European Community (EC) was the predecessor to the current European Union (EU), formed in 1957 by six founding nations: Belgium, France, Germany (West), Italy, Luxembourg, and the Netherlands
df.filter(df.Country == 'European Community').count()

197

In [ ]:
#RSA -> The Republic of South Africa
df.filter(df.Country == 'RSA').count()

145

In [ ]:
#Éire is the Irish language name for the country of Ireland
df.filter(df.Country == 'EIRE').count()

23057

**Data Cleaning & Standardization**

In [ ]:
#remove duplicates
df = df.dropDuplicates()

In [ ]:
#updated total count
df.count()

2242062

In [ ]:
#distinct count
df.distinct().count()

2242062

Both count() and distinct count() are giving same result. hence duplicates are removed

In [ ]:
#filter rows with unitprice and quantity negative and zero
df = df.filter((df.Quantity > 0) & (df.UnitPrice > 0))

In [ ]:
#count of rows with unitprice and quantity negative and zero
df.filter((df.Quantity <= 0) & (df.UnitPrice <= 0)).count()

0

No records remain with zero or negative quantity or unit price after filtering

In [ ]:
#updated total count
df.count()

722643

In [ ]:
#filter already removes null but just to reconfirm dropping rows with missing qty
df = df.dropna(subset = ['Quantity'])

In [ ]:
#count of null values in Quantity
df.filter(df.Quantity.isNull()).count()

0

Count of null values in Quantity is zero ensure no nulls in Quantity column

In [ ]:
#updated total count
df.count()

722643

In [ ]:
#fill missing customer id as 'Guest'
df = df.fillna('Guest', subset=['CustomerID'])

In [ ]:
#count of missing customer id
df.filter(df.CustomerID.isNull()).count()

329951

Count of missing customer id is 0, indicating no blank customer id

In [ ]:
#Standardizing countries
from pyspark.sql.functions import when
df = df.withColumn('Country', \
              when(col('Country') == 'RSA', 'South Africa')\
              .when(col('Country') == 'EIRE', 'Ireland')\
              .when(col('Country') == 'European Community', 'EU-Other')\
              .when(col('Country') == 'Unspecified', 'Unknown')\
              .otherwise(col('Country')))

In [ ]:
#Unspecified country count
df.filter(df.Country == 'Unspecified').count()

0

In [ ]:
#RSA count
df.filter(df.Country == 'RSA').count()

0

In [ ]:
#EIRE count
df.filter(df.Country == 'EIRE').count()

0

In [ ]:
#European Community count
df.filter(df.Country == 'European Community').count()

0

Count of rows with country Unspecified, RSA, EIRE, European Community is all 0 indicating successful standardization of country values.

In [ ]:
#final count after all data cleaning steps
df.count()

722643

After applying data quality rules, the dataset was reduced to ~723K high-quality transactional records suitable for analysis.

In [ ]:
df.show(10)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536370|    10002|INFLATABLE POLITI...|      48|2010-12-01 08:45:00|     0.85|   12583.0|        France|
|   536409|   85099B|JUMBO BAG RED RET...|       2|2010-12-01 11:45:00|     1.95|   17908.0|United Kingdom|
|   536488|    22376|AIRLINE BAG VINTA...|       1|2010-12-01 12:31:00|     4.25|   17897.0|United Kingdom|
|   536532|    22866|HAND WARMER SCOTT...|      12|2010-12-01 13:24:00|      2.1|   12433.0|        Norway|
|   536562|    22313|OFFICE MUG WARMER...|       6|2010-12-01 15:08:00|     2.95|   13468.0|United Kingdom|
|   536569|    22660|DOORMAT I LOVE LO...|       1|2010-12-01 15:35:00|     7.95|   16274.0|United Kingdom|
|   536572|    22695| WICKER

In [ ]:
#add revenue column
df = df.withColumn('Revenue', col('Quantity') * col('UnitPrice'))

In [ ]:
#confirming no null values in InvoiceDate
df.filter(df.InvoiceDate.isNull()).count()

0

In [ ]:
#Extracting year from Invoice date
from pyspark.sql.functions import year
df = df.withColumn('Year', year('InvoiceDate'))

In [ ]:
#Extracting month from Invoice date
from pyspark.sql.functions import month
df = df.withColumn('MonthNo', month('InvoiceDate'))

In [ ]:
#Extracting month name from Invoice date
from pyspark.sql.functions import monthname
df = df.withColumn('MonthName', monthname('InvoiceDate'))

In [ ]:
#creating year-month column for presentation whenever required.
from pyspark.sql.functions import date_format
df = df.withColumn('year_month', date_format('InvoiceDate', 'yyyy-MMM'))

In [ ]:
#caching the dataframe
df.cache()

DataFrame[InvoiceNo: string, StockCode: string, Description: string, Quantity: int, InvoiceDate: timestamp, UnitPrice: double, CustomerID: string, Country: string, Revenue: double, Year: int, MonthNo: int, MonthName: string, year_month: string]

The cleaned and enriched dataset is cached to optimize performance during repeated analytical operations.

In [ ]:
#display newly created columns
df.show(10)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+------------------+----+-------+---------+----------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|           Revenue|Year|MonthNo|MonthName|year_month|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+------------------+----+-------+---------+----------+
|   536370|    10002|INFLATABLE POLITI...|      48|2010-12-01 08:45:00|     0.85|   12583.0|        France|              40.8|2010|     12|      Dec|  2010-Dec|
|   536409|   85099B|JUMBO BAG RED RET...|       2|2010-12-01 11:45:00|     1.95|   17908.0|United Kingdom|               3.9|2010|     12|      Dec|  2010-Dec|
|   536488|    22376|AIRLINE BAG VINTA...|       1|2010-12-01 12:31:00|     4.25|   17897.0|United Kingdom|              4.25|2010|     12|      Dec|  2010-Dec|
|   536532|    22866|HAND WARMER S

In [ ]:
#Distinct years
df.select(['Year', 'MonthNo']).distinct().orderBy('Year', 'MonthNo').show()

+----+-------+
|Year|MonthNo|
+----+-------+
|2010|     12|
|2011|      1|
|2011|      2|
|2011|      3|
|2011|      4|
|2011|      5|
|2011|      6|
|2011|      7|
|2011|      8|
|2011|      9|
|2011|     10|
|2011|     11|
|2011|     12|
+----+-------+



The dataset spans a 13-month period from December 2010 to December 2011

**Business Analysis & Insights**

In [ ]:
#formatting function
from pyspark.sql.functions import format_number, lit, concat

def format_amount(col_name):
  return when(col(col_name) >= 1000000000, concat(format_number((col(col_name)/1000000000), 2), lit('B')))\
  .when(col(col_name) >= 1000000, concat(format_number((col(col_name)/1000000), 2), lit('M')))\
  .when(col(col_name) >= 1000, concat(format_number((col(col_name)/1000),2), lit('K')))\
  .otherwise(format_number(col(col_name), 2))

In [ ]:
#Main KPI's
#Total Orders
from pyspark.sql.functions import countDistinct
tot_orders = df.agg(countDistinct('InvoiceNo').alias('Total Orders'))
tot_orders.withColumn('Total Orders', format_amount('Total Orders')).show()

+------------+
|Total Orders|
+------------+
|      19.96K|
+------------+



In [ ]:
#Total Revenue
from pyspark.sql.functions import sum
tot_rev = df.agg(sum('Revenue').alias('Total Revenue'))
tot_rev.withColumn('Total Revenue', format_amount('Total Revenue')).show()

+-------------+
|Total Revenue|
+-------------+
|       15.20M|
+-------------+



In [ ]:
#Total Quantity
tot_qty = df.agg(sum('Quantity').alias('Total Quantity'))
tot_qty.withColumn('Total Quantity', format_amount('Total Quantity')).show()

+--------------+
|Total Quantity|
+--------------+
|         8.23M|
+--------------+



In [ ]:
#Average Order Value
aov = df.agg((sum('Revenue')/countDistinct('InvoiceNo')).alias('Average Order Value'))
aov.withColumn('Average Order Value', format_amount('Average Order Value')).show()

+-------------------+
|Average Order Value|
+-------------------+
|             761.60|
+-------------------+



In [ ]:
#Best Selling Product
grp_prod = df.groupBy('StockCode', 'Description').agg(sum('Quantity').alias('Total Quantity')).orderBy(col('Total Quantity').desc()).limit(1)
grp_prod.withColumn('Total Quantity', format_amount('Total Quantity')).show()

+---------+--------------------+--------------+
|StockCode|         Description|Total Quantity|
+---------+--------------------+--------------+
|    23843|PAPER CRAFT , LIT...|       161.99K|
+---------+--------------------+--------------+



The best-selling product by volume is PAPER CRAFT…, with ~162K units sold.

In [ ]:
#Top 3 Best Customers
grp_cust = df.groupBy('CustomerID').agg(sum('Revenue').alias('Total Revenue')).orderBy(col('Total Revenue').desc()).limit(3)
grp_cust.withColumn('Total Revenue', format_amount('Total Revenue')).show()

+----------+-------------+
|CustomerID|Total Revenue|
+----------+-------------+
|     Guest|        6.31M|
|   14646.0|      280.21K|
|   18102.0|      259.66K|
+----------+-------------+



Guest customers contribute a disproportionately high share of revenue, indicating strong anonymous or unregistered purchases.

In [ ]:
#Best Selling Country
grp_ctry = df.groupBy('Country').agg(sum('Quantity').alias('Total Quantity')).orderBy(col('Total Quantity').desc()).limit(1)
grp_ctry.withColumn('Total Quantity', format_amount('Total Quantity')).show()

+--------------+--------------+
|       Country|Total Quantity|
+--------------+--------------+
|United Kingdom|         6.86M|
+--------------+--------------+



The United Kingdom is the top-selling country by volume, indicating the highest number of items sold.

In [ ]:
#Highest revenue generated month
grp_mon = df.groupBy('year_month').agg(sum('Revenue').alias('Total Revenue')).orderBy(col('Total Revenue').desc()).limit(1)
grp_mon.withColumn('Total Revenue', format_amount('Total Revenue')).show()

+----------+-------------+
|year_month|Total Revenue|
+----------+-------------+
|  2011-Nov|        2.08M|
+----------+-------------+



In [ ]:
#Business Analysis
#Revenue by products
rev_prod = df.groupBy('StockCode', 'Description').agg(sum('Revenue').alias('Total Revenue')).orderBy(col('Total Revenue').desc())
rev_prod.withColumn('Total Revenue', format_amount('Total Revenue')).show(10)

+---------+--------------------+-------------+
|StockCode|         Description|Total Revenue|
+---------+--------------------+-------------+
|    23843|PAPER CRAFT , LIT...|      336.94K|
|    22423|REGENCY CAKESTAND...|      255.39K|
|      DOT|      DOTCOM POSTAGE|      213.63K|
|   85123A|WHITE HANGING HEA...|      162.80K|
|    23166|MEDIUM CERAMIC TO...|      161.29K|
|   85099B|JUMBO BAG RED RET...|      137.13K|
|    47566|       PARTY BUNTING|      133.96K|
|        M|              Manual|      115.26K|
|     POST|             POSTAGE|      111.98K|
|    23084|  RABBIT NIGHT LIGHT|       90.44K|
+---------+--------------------+-------------+
only showing top 10 rows


In [ ]:
#Revenue by country
rev_ctry = df.groupBy('Country').agg(sum('Revenue').alias('Total Revenue')).orderBy(col('Total Revenue').desc())
rev_ctry.withColumn('Total Revenue', format_amount('Total Revenue')).show(10)

+---------------+-------------+
|        Country|Total Revenue|
+---------------+-------------+
| United Kingdom|       12.77M|
|    Netherlands|      420.44K|
|        Ireland|      412.74K|
|        Germany|      344.65K|
|         France|      317.84K|
|      Australia|      208.34K|
|          Spain|       90.78K|
|    Switzerland|       83.79K|
|        Belgium|       62.06K|
|         Sweden|       57.16K|
|         Norway|       53.84K|
|          Japan|       51.69K|
|       Portugal|       50.21K|
|      Singapore|       37.89K|
|        Finland|       33.55K|
|Channel Islands|       30.68K|
|        Denmark|       28.75K|
|          Italy|       26.54K|
|         Cyprus|       20.30K|
|      Hong Kong|       15.48K|
+---------------+-------------+
only showing top 20 rows


In [ ]:
#Revenue by customers
rev_cust = df.groupBy('CustomerID').agg(sum('Revenue').alias('Total Revenue')).orderBy(col('Total Revenue').desc())
rev_cust.withColumn('Total Revenue', format_amount('Total Revenue')).show(10)

+----------+-------------+
|CustomerID|Total Revenue|
+----------+-------------+
|     Guest|        6.31M|
|   14646.0|      280.21K|
|   18102.0|      259.66K|
|   17450.0|      194.39K|
|   16446.0|      168.47K|
|   14911.0|      143.71K|
|   12415.0|      124.91K|
|   14156.0|      117.21K|
|   17511.0|       91.06K|
|   16029.0|       80.85K|
|   12346.0|       77.18K|
|   16684.0|       66.65K|
|   14096.0|       65.16K|
|   13694.0|       65.04K|
|   15311.0|       60.63K|
|   13089.0|       58.76K|
|   17949.0|       58.51K|
|   15769.0|       56.25K|
|   15061.0|       54.53K|
|   14298.0|       51.53K|
+----------+-------------+
only showing top 20 rows


In [ ]:
#Sales trend
rev_trend = df.groupBy('year', 'MonthNo').agg(sum('Revenue').alias('Total Revenue')).orderBy(col('Year'), col('MonthNo'))
rev_trend.withColumn('Total Revenue', format_amount('Total Revenue')).show()

+----+-------+-------------+
|year|MonthNo|Total Revenue|
+----+-------+-------------+
|2010|     12|        1.12M|
|2011|      1|        1.01M|
|2011|      2|      742.80K|
|2011|      3|        1.02M|
|2011|      4|      780.43K|
|2011|      5|        1.10M|
|2011|      6|        1.05M|
|2011|      7|        1.01M|
|2011|      8|        1.09M|
|2011|      9|        1.53M|
|2011|     10|        1.69M|
|2011|     11|        2.08M|
|2011|     12|      982.47K|
+----+-------+-------------+



In [ ]:
#Top Performing Products per Country
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
windowSpec = Window.partitionBy('Country').orderBy(col('Total Revenue').desc())
grp_rev = df.groupBy('Country', 'StockCode').agg(sum('Revenue').alias('Total Revenue'))
top_rank = grp_rev.withColumn('RankNo', row_number().over(windowSpec)).filter(col('RankNo') == 1)
top_rank.withColumn('Total Revenue', format_amount('Total Revenue')).show()

+---------------+---------+-------------+------+
|        Country|StockCode|Total Revenue|RankNo|
+---------------+---------+-------------+------+
|      Australia|    23084|        5.72K|     1|
|        Austria|     POST|        2.11K|     1|
|        Bahrain|   72802B|       231.24|     1|
|        Belgium|     POST|        6.38K|     1|
|         Brazil|    22423|       350.40|     1|
|         Canada|     POST|       550.94|     1|
|Channel Islands|    22423|        1.04K|     1|
|         Cyprus|    22827|       870.00|     1|
| Czech Republic|    22231|       104.40|     1|
|        Denmark|    22624|        1.39K|     1|
|       EU-Other|     POST|       237.00|     1|
|        Finland|     POST|        5.17K|     1|
|         France|     POST|       24.04K|     1|
|        Germany|     POST|       30.95K|     1|
|         Greece|     POST|       570.00|     1|
|      Hong Kong|        M|        5.56K|     1|
|        Iceland|   84558A|       690.30|     1|
|        Ireland|   

In [ ]:
#Top Selling Products per Country
window_spec = Window.partitionBy('Country').orderBy(col('Total Quantity').desc())
grp_qty = df.groupBy('Country', 'StockCode').agg(sum('Quantity').alias('Total Quantity'))
top_qty = grp_qty.withColumn('RankNo', row_number().over(window_spec)).filter(col('RankNo') == 1)
top_qty.withColumn('Total Quantity', format_amount('Total Quantity')).show()

+---------------+---------+--------------+------+
|        Country|StockCode|Total Quantity|RankNo|
+---------------+---------+--------------+------+
|      Australia|    22492|         4.14K|     1|
|        Austria|    21918|        576.00|     1|
|        Bahrain|    23076|         96.00|     1|
|        Belgium|    21212|        768.00|     1|
|         Brazil|    22698|         48.00|     1|
|         Canada|    84077|        576.00|     1|
|Channel Islands|    21785|        814.00|     1|
|         Cyprus|    84568|        576.00|     1|
| Czech Republic|    22579|        144.00|     1|
|        Denmark|    21915|        576.00|     1|
|       EU-Other|    22572|         48.00|     1|
|        Finland|   84997D|        772.00|     1|
|         France|    23084|         6.10K|     1|
|        Germany|    15036|         2.21K|     1|
|         Greece|    21616|         96.00|     1|
|      Hong Kong|    22326|        150.00|     1|
|        Iceland|    23076|        480.00|     1|


In [ ]:
#Average order value by country
aov_ctry = df.groupBy('Country').agg((sum('Revenue')/countDistinct('InvoiceNo')).alias('Avg Order Value')).orderBy(col('Avg Order Value').desc())
aov_ctry.withColumn('Avg Order Value', format_amount('Avg Order Value')).show()

+--------------------+---------------+
|             Country|Avg Order Value|
+--------------------+---------------+
|           Singapore|          5.41K|
|         Netherlands|          4.47K|
|           Australia|          3.66K|
|               Japan|          2.72K|
|             Lebanon|          2.33K|
|              Brazil|          1.69K|
|             Denmark|          1.60K|
|              Sweden|          1.59K|
|         Switzerland|          1.55K|
|              Israel|          1.51K|
|        South Africa|          1.50K|
|              Norway|          1.50K|
|              Greece|          1.46K|
|             Ireland|          1.43K|
|           Hong Kong|          1.41K|
|              Cyprus|          1.27K|
|     Channel Islands|          1.18K|
|                 USA|          1.07K|
|               Spain|          1.01K|
|United Arab Emirates|         982.42|
+--------------------+---------------+
only showing top 20 rows


Countries with lower order volumes tend to have higher average order values, indicating premium purchasing behavior.

In [ ]:
#Parquet Persistence
df.write.mode("overwrite").parquet("/content/online_retail_cleaned_parquet")

The cleaned dataset was persisted in Parquet format to optimize analytical performance.

**Key Insights**

A total of ~**20K** orders were placed during the analysis period, generating **15.2M** in revenue from **8.23M** units sold, indicating high transaction volume and strong overall sales performance.

The **best-selling product** by both quantity and revenue is StockCode **23843 (PAPER CRAFT, LIT…)**, with approximately 162K units sold, contributing **336.94K** in revenue, making it a key revenue-driving product.

**Guest customers** contribute a disproportionately high share of total revenue **(6.31M)**, suggesting strong anonymous or unregistered purchasing behavior. Among registered customers, **Customer 14646** and **Customer 18102** are top contributors, generating **280.21K** and **259.66K** in revenue respectively.

**The United Kingdom** is the **top-performing country**, leading both in sales volume and revenue, with **12.77M** in total revenue, highlighting its dominance as the primary market.

Monthly revenue analysis shows a clear peak in **November 2011** (**2.08M** revenue), indicating seasonal demand, likely driven by festive or year-end shopping patterns.

**Conclusion**

This project showcases how PySpark can be effectively used to process large datasets and extract actionable business insights through scalable data analysis.